# metabeta — introductory demo

This notebook demonstrates the end-to-end inference workflow:
1. Load a real hierarchical dataset
2. Specify a mixed-effects formula
3. Package model checkpoints into a joint checkpoint
4. Run amortized posterior inference with `Router`

In [1]:
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Markdown, display

from metabeta.models.router import Router
from metabeta.utils.router import joinCheckpoints

## 1. Load the dataset

The `sleepstudy` dataset (Belenky et al. 2003) measures reaction times (ms) for 18 subjects over 10 days of sleep restriction. It is a classic two-level dataset: observations nested within subjects.

In [2]:
DATASET_PATH = Path('../metabeta/datasets/from-r/parquet/sleep.parquet')

df = pd.read_parquet(DATASET_PATH)
print(f'{len(df)} observations, {df["group"].nunique()} subjects')
df.head()

180 observations, 18 subjects


,y,Days,group
0,249.5600,0.0,308
1,258.7047,1.0,308
2,250.8006,2.0,308
3,321.4398,3.0,308
4,356.8519,4.0,308


## 2. Specify a formula

We use a standard lme4-style formula. `Days` enters as a fixed slope; a random intercept is estimated per subject.

In [3]:
FORMULA = 'y ~ Days + (1 | group)'

## 3. Build a joint checkpoint

`Router` expects a *joint checkpoint* — a single `.pt` file that bundles one or more trained submodels with their routing metadata. `joinCheckpoints` assembles it from individual training checkpoints.

Here we use a single `small-n-mixed` submodel, trained on continuous (normally-distributed) outcomes.

In [4]:
CHECKPOINT_DIR = Path('../metabeta/outputs/checkpoints')
JOINT_PATH = Path('/tmp/joint_normal_v1.pt')

joinCheckpoints(
    {'small-n-s1': CHECKPOINT_DIR / 'data=small-n-mixed_model=large_seed=1'},
    output_path=JOINT_PATH,
)
print('Joint checkpoint written to', JOINT_PATH)

Joint checkpoint written to /tmp/joint_normal_v1.pt


## 4. Load the Router

`Router` loads the joint checkpoint and lazily instantiates submodels on first use.

In [5]:
router = Router(JOINT_PATH, device='cpu')
print('Submodels:', [s['id'] for s in router.submodels])

Submodels: ['small-n-s1']


## 5. Run inference

`router.sample()` accepts a raw DataFrame. It runs the `DataPreprocessor` internally (`fit_preprocessor=True`), builds design matrices from the formula, routes the dataset to the compatible submodel, and returns posterior samples.

The preprocessor z-scores both `y` and the continuous predictors `X`. Posterior samples are returned on the **standardized X, original y** scale by default:
- Fixed effects (`ffx`): slopes are Δy per SD of each predictor; the intercept is the expected y at the mean of each predictor.
- Random-effect SDs (`sigma_rfx`) and residual SD (`sigma_eps`) are in the original units of `y`.

Passing `diagnostics=True` additionally computes the posterior predictive, from which R², LOO-NLL, and Pareto k are derived.

In [6]:
result = router.sample(
    df,
    formula=FORMULA,
    fit_preprocessor=True,
    n_samples=1000,
    diagnostics=True,
)

print('Routed to submodel:', result.routes)
print('ffx samples shape  (B, S, d):', result.proposal.ffx.shape)
print('rfx samples shape  (B, m, S, q):', result.proposal.rfx.shape)
print('sigma_rfx shape    (B, S, q):', result.proposal.sigma_rfx.shape)
print('sigma_eps shape    (B, S):', result.proposal.sigma_eps.shape)

Routed to submodel: ['small-n-s1']
ffx samples shape  (B, S, d): torch.Size([1, 1000, 4])
rfx samples shape  (B, m, S, q): torch.Size([1, 18, 1000, 2])
sigma_rfx shape    (B, S, q): torch.Size([1, 1000, 2])
sigma_eps shape    (B, S): torch.Size([1, 1000])


## 6. Inspect posterior summaries

`posteriorSummary` prints fixed effects (with posterior contraction), variance components, and fit diagnostics.

By default (`x_scale='standardized'`), fixed-effect priors are shown on the fully standardized (★) scale, while estimates are on the **standardized X, original y** scale. Pass `x_scale='original'` to back-transform everything to the original units of `y` and `X`.

`rfxTable` gives the full per-group random effects table in original y units.

In [7]:
print(router.posteriorSummary(result))

Formula:  y ~ Days + (1 | group)
Scale:    standardized X, original y  (slopes: Δy per SD predictor; σ and rfx: y units)
Priors (★):
  Intercept ~ N(0, 2.5)
  days ~ N(0, 2.5)
  σ_Intercept ~ HN(2.5)

Fixed Effects:
|           |   Mean |    SD |    2.5% |   97.5% |   P(>0) |   Contr. |
|:----------|-------:|------:|--------:|--------:|--------:|---------:|
| Intercept | -0.078 | 9.860 | -19.592 |  19.890 |   0.494 |    0.995 |
| days      | 30.510 | 4.569 |  21.831 |  39.503 |   1.000 |    0.999 |

Standard Deviations (y units):
|           |   Mean |    SD |   2.5% |   97.5% |
|:----------|-------:|------:|-------:|--------:|
| Intercept | 40.593 | 8.244 | 27.816 |  60.698 |
| Residual  | 31.179 | 1.722 | 27.997 |  34.727 |

n_samples = 1000   R² = 0.726   LOO-NLL = 0.907   Pareto k = 0.191


### Original-scale summary

Pass `x_scale='original'` to back-transform fixed effects to the raw units of `y` and `X` (Δy per unit predictor, intercept at `X=0`).

In [8]:
print(router.posteriorSummary(result, x_scale='original'))

Formula:  y ~ Days + (1 | group)
Scale:    original  (slopes: Δy per unit predictor; σ and rfx: y units)
Priors:
  Intercept ~ N(298.5, 140.4)
  days ~ N(0, 48.89)
  σ_Intercept ~ HN(140.4)

Fixed Effects:
|           |    Mean |     SD |    2.5% |   97.5% |   P(>0) |   Contr. |
|:----------|--------:|-------:|--------:|--------:|--------:|---------:|
| Intercept | 250.630 | 12.515 | 225.877 | 274.651 |   1.000 |    0.992 |
| days      |  10.622 |  1.591 |   7.601 |  13.753 |   1.000 |    0.999 |

Standard Deviations (y units):
|           |   Mean |    SD |   2.5% |   97.5% |
|:----------|-------:|------:|-------:|--------:|
| Intercept | 40.593 | 8.244 | 27.816 |  60.698 |
| Residual  | 31.179 | 1.722 | 27.997 |  34.727 |

n_samples = 1000   R² = 0.726   LOO-NLL = 0.907   Pareto k = 0.191


In [9]:
print(router.rfxTable(result))

Random Effects:
Scale:    original y units  (intercept deviations from population intercept β₀)
|   Group |   Intercept Mean |   Intercept 2.5% |   Intercept 97.5% |
|---------|------------------|------------------|-------------------|
|     308 |           41.439 |           16.782 |            66.852 |
|     309 |          -77.995 |         -103.834 |           -53.346 |
|     310 |          -63.688 |          -89.738 |           -35.609 |
|     330 |            4.411 |          -22.101 |            28.110 |
|     331 |            9.985 |          -16.264 |            34.098 |
|     332 |            8.144 |          -18.604 |            33.676 |
|     333 |           16.602 |           -9.234 |            41.715 |
|     334 |           -3.100 |          -29.074 |            23.172 |
|     335 |          -45.208 |          -71.907 |           -20.507 |
|     337 |           72.210 |           46.957 |            97.944 |
|     349 |          -21.218 |          -48.100 |             4.

In [10]:
import numpy as np
import statsmodels.formula.api as smf
from tabulate import tabulate

# fit the same model with REML
lmm = smf.mixedlm("y ~ Days", df, groups=df["group"]).fit(reml=True)

# fixed effects table
ci = lmm.conf_int()
fe_rows = [
    [name, lmm.fe_params[name], lmm.bse[name],
     ci.loc[name, 0], ci.loc[name, 1], lmm.tvalues[name], lmm.pvalues[name]]
    for name in lmm.fe_params.index
]
fe_table = tabulate(
    fe_rows, headers=["", "Coef.", "SE", "2.5%", "97.5%", "z", "p"],
    floatfmt=".3f", tablefmt="github",
)

# variance components
sigma_re  = np.sqrt(float(lmm.cov_re.values[0, 0]))
sigma_eps = np.sqrt(lmm.scale)
sd_table = tabulate(
    [["Intercept", sigma_re], ["Residual", sigma_eps]],
    headers=["", "Std.Dev."], floatfmt=".3f", tablefmt="github",
)

# Nakagawa & Schielzeth (2013) R²
fe_var  = float(np.var(lmm.model.exog @ lmm.fe_params.values))
re_var  = float(lmm.cov_re.values[0, 0])
res_var = float(lmm.scale)
r2_m = fe_var / (fe_var + re_var + res_var)
r2_c = (fe_var + re_var) / (fe_var + re_var + res_var)

display(Markdown(
    "Fixed Effects:\n\n" + fe_table
    + "\n\nStandard Deviations:\n\n" + sd_table
    + f"\n\nMarginal R² = {r2_m:.3f}   Conditional R² = {r2_c:.3f}"
))

Fixed Effects:

|           |   Coef. |    SE |    2.5% |   97.5% |      z |     p |
|-----------|---------|-------|---------|---------|--------|-------|
| Intercept | 251.405 | 9.747 | 232.302 | 270.508 | 25.794 | 0.000 |
| Days      |  10.467 | 0.804 |   8.891 |  12.044 | 13.015 | 0.000 |

Standard Deviations:

|           |   Std.Dev. |
|-----------|------------|
| Intercept |     37.124 |
| Residual  |     30.991 |

Marginal R² = 0.279   Conditional R² = 0.704